In [1]:
# !pip install transformers==4.26 sentencepiece
from transformers import pipeline
pipe = pipeline(model="kubota/luke-large-defamation-detection-japanese")
pipe("あの人は殺人を犯した犯罪者らしい")


config.json: 0.00B [00:00, ?B/s]

c:\Users\11037\miniconda3\envs\rl_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\11037\.cache\huggingface\hub\models--kubota--luke-large-defamation-detection-japanese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/842k [00:00<?, ?B/s]

entity_vocab.json:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

[{'label': '名誉を低下させる発言', 'score': 0.788667619228363}]

In [2]:
pipe("私はどんな人だと思う？")

[{'label': '中傷性のない発言', 'score': 0.9288204908370972}]

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from huggingface_hub import hf_hub_download

model_name = "yukismd/HateSpeechClassification-japanese-gpt-neox-3-6b-instruction-ppo"  # either local folder or huggingface model name
hf_hub_download(repo_id=model_name, filename="classification_head.pth", local_dir="./")

device = "cuda" if torch.cuda.is_available() else "cpu"

# either local folder or huggingface model name
# Important: The prompt needs to be in the same format the model was trained with.
# You can find an example prompt in the experiment logs.
prompt = "女なんてまともなやつはいないぞ 彼女なんて絶対に作るもんじゃない"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,   # 强制全精度
    device_map={"": device},      # 绑定到 GPU/CPU
    trust_remote_code=True,
).eval()


# 1. 加载分类头
head_weights = torch.load("classification_head.pth", map_location=device)
head = torch.nn.Linear(1, 1, bias=False).to(device)
head.weight.data = head_weights

# 2. 文本编码
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)

# 3. 模型推理 + 分类头
out = model(**inputs).logits
out = out.to(head.weight.dtype)   # dtype 对齐
logits = head(out[:, -1])

print(logits)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tensor([[9.9597]], grad_fn=<MmBackward0>)


In [ ]:
import torch
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("kit-nlp/electra-small-japanese-discriminator-cyberbullying")
model = AutoModelForSequenceClassification.from_pretrained("kit-nlp/electra-small-japanese-discriminator-cyberbullying",trust_remote_code=True)

prompt = "女なんてまともなやつはいないぞ 彼女なんて絶対に作るもんじゃない"

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

result = model(**inputs).logits

pred_label = torch.argmax(result, dim=-1).item()
print(int(pred_label))


1


In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ptaszynski/yacis-electra-small-japanese-cyberbullying")
model = AutoModelForSequenceClassification.from_pretrained("ptaszynski/yacis-electra-small-japanese-cyberbullying",trust_remote_code=True)

prompt = "女なんてまともなやつはいないぞ 彼女なんて絶対に作るもんじゃない"

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

result = model(**inputs).logits

pred_label = torch.argmax(result, dim=-1).item()
print(int(pred_label))

tokenizer_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

c:\Users\11037\miniconda3\envs\rl_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\11037\.cache\huggingface\hub\models--ptaszynski--yacis-electra-small-japanese-cyberbullying. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\11037\miniconda3\envs\rl_env\Lib\site-packages\huggingface_hub\file_

config.json:   0%|          | 0.00/810 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/55.0M [00:00<?, ?B/s]

1


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1. Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
tokenizer.pad_token = tokenizer.eos_token

# 2. Load the DuoGuard-0.5B model
model = AutoModelForSequenceClassification.from_pretrained(
    "DuoGuard/DuoGuard-1.5B-transfer",
    torch_dtype=torch.bfloat16
).to('cpu')

# 3. Define a sample prompt to test
prompt = "女なんてまともなやつはいないぞ 彼女なんて絶対に作るもんじゃない"

# 4. Tokenize the prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt", 
    truncation=True, 
    max_length=512  # adjust as needed
).to('cpu')

# 5. Run the model (inference)
with torch.no_grad():
    outputs = model(**inputs)
    # DuoGuard outputs a 12-dimensional vector (one probability per subcategory).
    logits = outputs.logits  # shape: (batch_size, 12)
    probabilities = torch.sigmoid(logits)  # element-wise sigmoid

# 6. Multi-label predictions (one for each category)
threshold = 0.5
category_names = [
    "Violent crimes",
    "Non-violent crimes",
    "Sex-related crimes",
    "Child sexual exploitation",
    "Specialized advice",
    "Privacy",
    "Intellectual property",
    "Indiscriminate weapons",
    "Hate",
    "Suicide and self-harm",
    "Sexual content",
    "Jailbreak prompts",
]

prob_vector = probabilities[0].tolist()

# 找出最大概率所在的索引
max_index = prob_vector.index(max(prob_vector))

# “Hate”的索引
hate_index = category_names.index("Hate")

# 如果最大概率的类别刚好是“Hate”，返回1，否则返回0
if max_index == hate_index:
    print(1)
else:
    print(0)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


1
